# Task 2. `eth_getLogs` Transfer 샘플 검증

## 검증 의도

- 현재 Python RPC client로 `eth_getLogs` 호출이 가능한지 확인하고, ERC-20 Transfer event의 `topic0`, `topic1`, `topic2`, `data` 해석 흐름을 DataFrame으로 확인합니다.

- 이 노트북 파일은 수집 파이프라인 전체 실행이 아니라 provider 응답과 decode 규칙을 눈으로 검증하는 smoke 자료입니다.

## 검증 경계

- sample contract는 WETH입니다. 과제의 Tether Treasury 집계 범위를 넓힌다는 의미가 아닙니다.
- 실제 수집 범위는 `PipelineSettings.collection_scope`와 Airflow DAG 기준입니다.
- 외부 RPC credential이 없으면 `BLOCKED`로 남깁니다.

In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    """노트북 실행 위치와 무관하게 현재 repository root를 찾는다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "cryptoquant_pipeline").exists():
            return candidate
    raise RuntimeError("repository root를 찾지 못함. pyproject.toml과 src/cryptoquant_pipeline 확인 필요함.")


PROJECT_ROOT = find_repo_root()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)

SAFE_ENV_STATUS = {
    "project_root": str(PROJECT_ROOT),
    "env_file_exists": ENV_PATH.exists(),
    "eth_rpc_url_configured": bool(os.getenv("ETH_RPC_URL") or os.getenv("CHAINSTACK_RPC_URL")),
    "auth_mode": os.getenv("ETH_RPC_AUTH_MODE") or os.getenv("CHAINSTACK_AUTH_MODE") or "none",
}

display(Markdown("### Repository 및 환경변수 확인"))
display(SAFE_ENV_STATUS)

### Repository 및 환경변수 확인

{'project_root': '/workspace',
 'env_file_exists': True,
 'eth_rpc_url_configured': True,
 'auth_mode': 'none'}

In [3]:
from cryptoquant_pipeline.config import PipelineSettings
from cryptoquant_pipeline.exceptions import ConfigError

settings = None
settings_status = "READY"
settings_error = None
try:
    settings = PipelineSettings.from_env()
except ConfigError as exc:
    settings_status = "BLOCKED"
    settings_error = f"{exc.__class__.__name__}: {exc}"

safe_settings = {
    "status": settings_status,
    "error": settings_error,
    "chain_id": None if settings is None else settings.chain_id,
    "auth_mode": None if settings is None else settings.provider.auth_mode,
    "provider_configured": settings is not None,
    "max_blocks_per_log_request": None if settings is None else settings.max_blocks_per_log_request,
    "requests_per_second": None if settings is None else settings.rpc_requests_per_second,
    "delta_logs_path": None if settings is None else str(settings.delta_logs_path),
    "duckdb_path": None if settings is None else str(settings.duckdb_path),
    "collection_scope": None if settings is None else settings.collection_scope.scope_id,
    "collection_scope_fingerprint": None if settings is None else settings.collection_scope.fingerprint,
}

display(Markdown("### Python 설정 객체 검증"))
display(safe_settings)

### Python 설정 객체 검증

{'status': 'READY',
 'error': None,
 'chain_id': 1,
 'auth_mode': 'none',
 'provider_configured': True,
 'max_blocks_per_log_request': 10,
 'requests_per_second': 4.0,
 'delta_logs_path': '/opt/airflow/data/delta/ethereum_logs_v2',
 'duckdb_path': '/opt/airflow/data/analytics/ethereum_analytics_v2.duckdb',
 'collection_scope': 'transfer_topic_all_addresses',
 'collection_scope_fingerprint': 'af0864e874141de6657364a36407791a0769dee5c55679b2089c4efb33e8b885'}

In [4]:
from decimal import Decimal

import pandas as pd

from cryptoquant_pipeline.config import TRANSFER_TOPIC0
from cryptoquant_pipeline.log_normalizer import amount_with_decimals, decode_uint256_decimal, topic_to_address
from cryptoquant_pipeline.rpc_client import EthereumJsonRpcClient

WETH_MAINNET_ADDRESS = "0xC02aaA39b223FE8D0A0E5C4F27eAD9083C756Cc2"
WETH_DECIMALS = 18
BLOCK_WINDOW = 10

if settings is None:
    display(Markdown("### eth_getLogs skipped\n`ETH_RPC_URL`이 없어 실제 provider 호출을 실행하지 않았습니다."))
    df_logs = pd.DataFrame()
else:
    with EthereumJsonRpcClient(
        settings.provider,
        timeout_seconds=settings.rpc_timeout_seconds,
        max_retries=settings.rpc_max_retries,
        requests_per_second=settings.rpc_requests_per_second,
    ) as client:
        finalized_block = client.eth_get_finalized_block()
        finalized_block_number = int(finalized_block["number"], 16)
        from_block = max(0, finalized_block_number - BLOCK_WINDOW + 1)
        to_block = finalized_block_number
        logs = client.eth_get_logs(
            from_block=from_block,
            to_block=to_block,
            address=WETH_MAINNET_ADDRESS,
            topics=[TRANSFER_TOPIC0],
        )

    records = []
    for log in logs:
        raw_amount = decode_uint256_decimal(str(log["data"]))
        records.append(
            {
                "block_number": int(str(log["blockNumber"]), 16),
                "transaction_hash": str(log["transactionHash"]),
                "log_index": int(str(log["logIndex"]), 16),
                "contract_address": str(log["address"]).lower(),
                "topic0": str(log["topics"][0]).lower(),
                "from_address": topic_to_address(str(log["topics"][1])),
                "to_address": topic_to_address(str(log["topics"][2])),
                "value_raw_decimal_text": str(raw_amount),
                "value_weth_decimal_text": str(amount_with_decimals(raw_amount, WETH_DECIMALS)),
            }
        )

    df_logs = pd.DataFrame(records)
    summary = {
        "from_block": from_block,
        "to_block": to_block,
        "contract": WETH_MAINNET_ADDRESS.lower(),
        "topic0": TRANSFER_TOPIC0,
        "log_count": len(df_logs),
        "unique_transaction_count": 0 if df_logs.empty else int(df_logs["transaction_hash"].nunique()),
    }
    display(Markdown("### eth_getLogs 호출 결과"))
    display(summary)
    display(df_logs.head(20))

    if not df_logs.empty and (df_logs["topic0"] != TRANSFER_TOPIC0).any():
        raise RuntimeError("[Transfer topic0]가 아닌 log가 포함됨.")

### eth_getLogs 호출 결과

{'from_block': 25374461,
 'to_block': 25374470,
 'contract': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'topic0': '0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef',
 'log_count': 369,
 'unique_transaction_count': 181}

,block_number,transaction_hash,log_index,contract_address,topic0,from_address,to_address,value_raw_decimal_text,value_weth_decimal_text
0,25374461,0x7e6b542fb059396a0cd33eab05f6b0d1fdec3e38b488...,1,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x000000000004444c5dc75cb358380d2e3de08a90,0x0000000aa232009084bd71a5797d089aa4edfad4,367002171395585988,0.367002171395585988
1,25374461,0x7e6b542fb059396a0cd33eab05f6b0d1fdec3e38b488...,5,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x0000000aa232009084bd71a5797d089aa4edfad4,0x73d53553ee552c1f2a9722e6407d43e41e19593f,233209577247375072,0.233209577247375072
2,25374461,0xacb1da00078de2d1853094991c362d192f3c9fa57704...,10,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x1f2f10d1c40777ae1da742455c65828ff36df387,0xc7bbec68d12a0d1830360f8ec58fa599ba1b0e9b,38354181791787838,0.038354181791787838
3,25374461,0xdd10e76d86b8d6b7dd028fadf34ab077b82871a006e5...,110,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x12d44f8b4d3e28bdeb35e8e2e6585dfcc33eda5f,0x111111125421ca6dc452d289314280a0f8842a65,64785497533657051,0.064785497533657051
4,25374461,0x84a012b5f3096f1107ac1cf3abac6511c30a715e9614...,128,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x69dffe2f5104cc7e28a948d9cbb61864e925895c,0x28b1dc1a5e3699a428bc51d234dfab7c9cb2a183,55847461919354333,0.055847461919354333
5,25374461,0x84a012b5f3096f1107ac1cf3abac6511c30a715e9614...,131,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x28b1dc1a5e3699a428bc51d234dfab7c9cb2a183,0x5703b683c7f928b721ca95da988d73a3299d4757,55847461919354333,0.055847461919354333
6,25374461,0xe905669b1de1a103374be0fd7015b82e8f5a8939c5fe...,141,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x66a9893cc07d91d95644aedd05d03f95e1dba8af,0xc7bbec68d12a0d1830360f8ec58fa599ba1b0e9b,1135666759950610,0.00113566675995061
7,25374461,0xb9f5e68c1f055574b8bc690da8df009c6c641045fdb2...,181,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0xf22fdd2be7c6da9788e4941a6ffc78ca99d7b15c,0xa43fe16908251ee70ef74718545e4fe6c5ccec9f,93282912844858544,0.093282912844858544
8,25374461,0x2049e14848bd1898423f55e9a19db94a92f701078e90...,218,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0xbee3211ab312a8d065c4fef0247448e17a8da000,0x8f10b468b06c6fd214b65f87778827f7d113f996,342341596127431700,0.3423415961274317
9,25374461,0x2049e14848bd1898423f55e9a19db94a92f701078e90...,220,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x8f10b468b06c6fd214b65f87778827f7d113f996,0x2c683fad51da2cd17793219cc86439c1875c353e,342341596127431700,0.3423415961274317
